In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import MessagesState, StateGraph, START, END
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.messages import HumanMessage
#from langchain_nvidia_ai_endpoints import ChatNVIDIA
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
#llm = init_chat_model(model="mistral:latest", model_provider="ollama", temperature=0.0)
load_dotenv(override=True)
#llm = ChatNVIDIA(model="nvidia/nemotron-3-nano-30b-a3b", temperature=0.0)
llm = init_chat_model(model="nvidia/nemotron-3-nano-30b-a3b", model_provider="nvidia", temperature=0.0)
CLASSIFIER_SYSTEM_PROMPT = """You are a routing assistant for an internal support desk. Classify each user message into exactly one department label.

Labels (pick exactly one):
- finance: budgets, invoices, expenses, accounting, taxes, reimbursement, purchase orders, payroll amounts
- hr: hiring, PTO, benefits, onboarding, performance reviews, workplace policies, employee relations
- technical: software, hardware, IT support, bugs, deployments, APIs, databases, programming, infrastructure, code changes
- others: messages that do not clearly belong to finance, hr, or technical

Rules:
- Use technical for software development, coding, or IT issues (e.g. Python, bugs, servers) even if money is mentioned in passing.
- Use hr only for people and workplace topics, not for tech tools unless the issue is HR policy.
- Use finance only for money, billing, and accounting topics—not for engineering work.
- When unsure between technical and others, prefer technical if the message mentions code, apps, systems, or engineering.
- Output only the structured label field; do not explain your reasoning."""

classifier_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(CLASSIFIER_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{user_msg}"),
])


In [ ]:
class ClassifierIntent(BaseModel):
    msg_intent: Literal["finance", "hr", "technical", "others"] = Field(
        ...,
        description="Department route: finance, hr, technical, or others.",
    )

class MsgState(MessagesState):
    intent: str | None

In [ ]:
def identify_the_process(state: MsgState):
    user_msg = state["messages"][-1].content
    llm_with_struct = llm.with_structured_output(ClassifierIntent)
    chain = classifier_prompt | llm_with_struct
    ident_response: ClassifierIntent = chain.invoke({"user_msg": user_msg})
    # ident_response:ClassifierIntent = llm_with_struct.invoke([
    #     {'role':'system', 'content':CLASSIFIER_SYSTEM_PROMPT},
    #     {'role':'user', 'content':user_msg
    # }])
    return {"intent": ident_response.msg_intent}

def hrm_process(state: MsgState):
    hr_msg = llm.invoke([{'role':'system','content':'You are the HR Process intelligent system'}] + state['messages'])
    return {"messages":[hr_msg]}

def accounting_process(state: MsgState):
    acc_msg = llm.invoke([{'role':'system','content':'You are the finance expert system'}] + state['messages'])
    return {"messages":[acc_msg]}

def tech_team_process(state: MsgState):
    tech_msg = llm.invoke([{'role':'system','content':'You are the Technical expert system'}] + state['messages'])
    return {"messages":[tech_msg]}

In [ ]:
wf_builder = StateGraph(MsgState)
wf_builder.add_node("classify_process", identify_the_process)
wf_builder.add_node("finance_process", accounting_process)
wf_builder.add_node("hrm_process", hrm_process)
wf_builder.add_node("tech_process", tech_team_process)

wf_builder.add_edge(START, "classify_process")
wf_builder.add_conditional_edges("classify_process", lambda state:state["intent"], {
    "finance":"finance_process",
    "hr":"hrm_process",
    "technical":"tech_process",
    "others":END
})
wf_builder.add_edge("finance_process", END)
wf_builder.add_edge("hrm_process", END)
wf_builder.add_edge("tech_process", END)

checkpointer  = InMemorySaver()
graph = wf_builder.compile(checkpointer=checkpointer)
_config = {"configurable":{"thread_id": "usr-classify-001"}}

### Streaming - values - Display full state snapshot after every node - Raw shape: full State dict every time.

In [ ]:
while True:
    u_input = str(input("User: Enter your Query: ")).strip()
    if u_input == "/bye":
        print("Thank you!")
        break
    res = graph.stream(input={
        "messages": HumanMessage(u_input)
    },config=_config, stream_mode="values")

    for chunk_data in res:
        print(chunk_data)

    ## Practice Checks
    #print(f"AI: f{res['messages'][-1].content}")
    #print(res)
    #for res_data, metadata in res:
        #node_name, update = next(iter(res_data.items()))
        #print(f"{node_name} - {update}")
        #print(res_data["messages"][-1].content)
        #print(res_data.content, end="")
        #print(f"\n {res_data.content}", end="")
        #print(f"\n {metadata}")
        #print(res_data['messages'][-1].content)
    # print(f"\n Classifier Intent: {res['intent']} \n")
    # for m in res['messages']:
    #     m.pretty_print()

### Streaming - Updates - only the delta each node returned - Raw shape: {node_name: partial_update} — one key per chunk.


In [ ]:
while True:
    u_input = str(input("User: Enter your Query: ")).strip()
    if u_input == "/bye":
        print("Thank you!")
        break
    res = graph.stream(input={
        "messages": HumanMessage(u_input)
    },config=_config, stream_mode="updates")

    for chunk_data in res:
        print(chunk_data)

### Streaming - messages - LLM token streaming, per-message metadata included - render as incremental text, buffer don't re-render per token
Raw shape: (AIMessageChunk, metadata_dict) tuple.

In [9]:
while True:
    u_input = str(input("User: Enter your Query: ")).strip()
    if u_input == "/bye":
        print("Thank you!")
        break
    res = graph.stream(input={
        "messages": HumanMessage(u_input)
    },config=_config, stream_mode="messages")

    for chunk_data, metadata in res:
        print(chunk_data.content, end="")

{"msg_intent": "technical"}## JIRA Access‑Request Process  
**Technical‑Expert View** – end‑to‑end workflow, roles, artefacts, automation hooks, and a ready‑to‑use checklist.  
*(The steps below assume you are using **Jira Service Management (JSM)** or **Jira Cloud** with a dedicated *Application‑Request* project. Adjust the naming of projects/boards to match your environment.)*  

---

### 1️⃣  Scope & Types of Access  

| Access Level | What It Grants | Typical Users | Default Permission Scheme |
|--------------|----------------|---------------|---------------------------|
| **Jira‑User (Basic)** | View public projects, create personal issues, comment. | All employees, contractors. | *Browse Projects* + *Create Issues* (if allowed). |
| **Jira‑Administrator** | Full system admin rights (manage users, schemes, plugins). | IT/DevOps, Security. | *Administrator* global permission. |
| **Project‑Specific Role** | *Administrator*, *Project‑Lead*, *Developer*, *Tester*, *Scrum‑Master*, *Vi

### Streaming - debug — render as raw structured logs, never end-user UI

In [10]:
while True:
    u_input = str(input("User: Enter your Query: ")).strip()
    if u_input == "/bye":
        print("Thank you!")
        break
    res = graph.stream(input={
        "messages": HumanMessage(u_input)
    },config=_config, stream_mode="debug")

    for res_log in res:
        print(res_log, end="")

{'step': 15, 'timestamp': '2026-07-02T14:03:56.244594+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'usr-classify-001', 'checkpoint_id': '1f1761ed-d7fe-604e-800f-41ad0cdf071e'}}, 'parent_config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'usr-classify-001', 'checkpoint_id': '1f1761e6-e86a-6484-800e-306d1e0dd093'}}, 'values': {'messages': [HumanMessage(content='employee onboarding process pls', additional_kwargs={}, response_metadata={}, id='cce9a07b-d010-4a00-b0d8-3e7aff75e521'), AIMessage(content='Below is a **complete, end‑to‑end Employee Onboarding Process** you can adopt (or adapt) for a typical organization.  \nIt is organized by **time‑frame**, **key activities**, **owners**, and **deliverables** so you can easily turn it into a checklist, a workflow in your HRIS, or a shared‑drive template.\n\n---\n\n## 1️⃣  Pre‑Boarding (0‑2\u202fWeeks Before Start Date)\n\n| Step | What Happens | Owner | Output / Artefact |\n|